# LeetCode 3745 — Maximize Expression of Three Elements bằng Python

Link đề: https://leetcode.com/problems/maximize-expression-of-three-elements/ · Độ khó: **Easy** · Chủ đề: Array, Greedy, Sorting, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def maximizeExpressionOfThree(self, nums: List[int]) -> int:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn ba phần tử ở **ba chỉ số khác nhau** `a, b, c` sao cho biểu thức `a + b - c` lớn nhất.
2. Biến bài thành **tham lam**: muốn `a + b` lớn nhất và `c` nhỏ nhất, nhưng ba chỉ số phải phân biệt.
3. Biết ba cách lấy "hai lớn nhất và một nhỏ nhất": sắp xếp, duyệt một lần theo dõi top-2 và min, và `heapq.nlargest/nsmallest`.
4. Viết 5 cách khác nhau (brute force ba vòng, sắp xếp, một lượt, `heapq`, bỏ phần tử đã dùng) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: dùng chính một phần tử hai lần, khởi tạo max bằng `0` khi có số âm, cộng `c` thay vì trừ, và chọn `c` là phần tử nhỏ nhất trước khi loại hai số lớn.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Hãy chọn ba phần tử `a`, `b`, `c` tại **ba chỉ số khác nhau** để giá trị `a + b - c` là **lớn nhất** có thể, rồi trả về giá trị đó.

"Ba chỉ số khác nhau" nghĩa là không được dùng một vị trí hai lần, nhưng giá trị có thể trùng nhau nếu chúng nằm ở các vị trí khác nhau.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,4,2,5]` | `8` | Chọn `a = 4`, `b = 5`, `c = 1`: `4 + 5 - 1 = 8` |
| 2 | `[-2,0,5,-2,4]` | `11` | Chọn `a = 5`, `b = 4`, `c = -2`: `5 + 4 - (-2) = 11` |

Constraints: `3 <= nums.length <= 100`, `-100 <= nums[i] <= 100`.

In [1]:
from itertools import permutations

for nums in [[1, 4, 2, 5], [-2, 0, 5, -2, 4]]:
    best = max(nums[i] + nums[j] - nums[k] for i, j, k in permutations(range(len(nums)), 3))
    print(f"nums={nums}  giá trị lớn nhất của a+b-c = {best}")

nums=[1, 4, 2, 5]  giá trị lớn nhất của a+b-c = 8
nums=[-2, 0, 5, -2, 4]  giá trị lớn nhất của a+b-c = 11


## 3. Kiến thức nền

### 3.1. Chỉ số khác nhau, giá trị có thể trùng

Chỉ số `i` là vị trí trong list; giá trị `nums[i]` là nội dung. Hai chỉ số khác nhau có thể cùng giá trị (ví dụ `-2` xuất hiện ở chỉ số 0 và 3 trong ví dụ 2). Bài yêu cầu chỉ số khác nhau, không yêu cầu giá trị khác nhau.

In [2]:
nums = [-2, 0, 5, -2, 4]
print([(i, v) for i, v in enumerate(nums)])
print(nums[0] == nums[3], 0 != 3)

[(0, -2), (1, 0), (2, 5), (3, -2), (4, 4)]
True True


### 3.2. `sorted` và slicing đầu/cuối

`sorted(nums)` trả list tăng dần. `a[-1]` và `a[-2]` là hai phần tử lớn nhất (ở **hai chỉ số khác nhau** của list đã sắp xếp), `a[0]` là nhỏ nhất.

In [3]:
a = sorted([-2, 0, 5, -2, 4])
print(a, "hai lớn nhất:", a[-1], a[-2], "nhỏ nhất:", a[0])
print(a[-2:], a[:1])

[-2, -2, 0, 4, 5] hai lớn nhất: 5 4 nhỏ nhất: -2
[4, 5] [-2]


### 3.3. `itertools.permutations`

`permutations(range(n), 3)` sinh mọi bộ ba chỉ số **đôi một khác nhau** có xét thứ tự: `n * (n-1) * (n-2)` bộ. Hợp với brute force; với `n = 100` là 970200 bộ, vẫn chạy được.

In [4]:
from itertools import permutations
print(list(permutations(range(3), 3)))
print(len(list(permutations(range(5), 3))), 5 * 4 * 3)

[(0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)]
60 60


### 3.4. `heapq.nlargest` và `heapq.nsmallest`

`heapq.nlargest(k, nums)` trả `k` phần tử lớn nhất (giảm dần); `heapq.nsmallest(k, nums)` trả `k` phần tử nhỏ nhất (tăng dần). Phần tử trùng giá trị được tính **riêng** theo vị trí.

In [5]:
import heapq
nums = [-2, 0, 5, -2, 4]
print(heapq.nlargest(2, nums), heapq.nsmallest(1, nums))
print(heapq.nlargest(3, [7, 7, 7]))

[5, 4] [-2]
[7, 7, 7]


### 3.5. Khởi tạo "âm vô cùng" khi tìm max

Khi tìm phần tử lớn nhất trong lúc duyệt, giá trị khởi tạo phải **nhỏ hơn mọi phần tử có thể**. Vì `nums[i]` có thể là số âm, khởi tạo bằng `0` là sai; dùng `float('-inf')` hoặc phần tử đầu tiên.

In [6]:
best = float('-inf')
for v in [-5, -3, -9]:
    best = max(best, v)
print(best)
wrong = 0
for v in [-5, -3, -9]:
    wrong = max(wrong, v)
print(wrong, "(sai: 0 không có trong mảng)")

-3
0 (sai: 0 không có trong mảng)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Biểu thức `a + b - c` có hai phần: **cộng hai số** và **trừ một số**. Để lớn nhất:

- `a` và `b` phải **lớn nhất có thể** (cộng vào càng lớn càng tốt);
- `c` phải **nhỏ nhất có thể** (trừ đi một số nhỏ, thậm chí âm, thì càng lợi).

Vì `a`, `b`, `c` phải ở ba chỉ số khác nhau, ta lấy **hai phần tử lớn nhất** cho `a, b` và **phần tử nhỏ nhất trong số còn lại** cho `c`. Do `n >= 3`, luôn còn ít nhất một phần tử cho `c`, và phần tử nhỏ nhất của cả mảng không nằm trong hai phần tử lớn nhất trừ khi tất cả bằng nhau (khi đó giá trị vẫn đúng).

Tính tay `[1, 4, 2, 5]`: hai lớn nhất `5, 4`, nhỏ nhất `1` -> `5 + 4 - 1 = 8`.

### Bảng tính tay

| nums | sắp xếp | `a`, `b` (hai lớn nhất) | `c` (nhỏ nhất) | kết quả |
|---|---|---|---|---|
| `[1,4,2,5]` | 1 2 4 5 | 5, 4 | 1 | 8 |
| `[-2,0,5,-2,4]` | -2 -2 0 4 5 | 5, 4 | -2 | 11 |
| `[3,3,3]` | 3 3 3 | 3, 3 | 3 | 3 |
| `[-1,-2,-3]` | -3 -2 -1 | -1, -2 | -3 | 0 |

In [7]:
for nums in [[1, 4, 2, 5], [-2, 0, 5, -2, 4], [3, 3, 3], [-1, -2, -3]]:
    a = sorted(nums)
    print(nums, "->", a[-1], "+", a[-2], "-", a[0], "=", a[-1] + a[-2] - a[0])

[1, 4, 2, 5] -> 5 + 4 - 1 = 8
[-2, 0, 5, -2, 4] -> 5 + 4 - -2 = 11
[3, 3, 3] -> 3 + 3 - 3 = 3
[-1, -2, -3] -> -1 + -2 - -3 = 0


## 5. Cách 1 — Brute Force / baseline

Thử **mọi bộ ba chỉ số khác nhau** `(i, j, k)`, tính `nums[i] + nums[j] - nums[k]`, giữ giá trị lớn nhất. Đúng vì duyệt hết mọi cách chọn hợp lệ. Với `n <= 100` (khoảng 10^6 bộ) vẫn chạy được.

Nút thắt: `O(n^3)` quá lãng phí; ta đã biết cách chọn tốt nhất là hai lớn nhất và một nhỏ nhất.

In [8]:
class SolutionBrute:
    def maximizeExpressionOfThree(self, nums):
        n = len(nums)
        best = None
        for i in range(n):
            for j in range(n):
                if j == i:
                    continue
                for k in range(n):
                    if k == i or k == j:
                        continue
                    val = nums[i] + nums[j] - nums[k]
                    if best is None or val > best:
                        best = val
        return best

print(SolutionBrute().maximizeExpressionOfThree([1, 4, 2, 5]), SolutionBrute().maximizeExpressionOfThree([-2, 0, 5, -2, 4]))

8 11


**Độ phức tạp:** Time `O(n^3)`, Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** tách biểu thức thành "hai số lớn nhất" và "một số nhỏ nhất" và chứng minh ba số đó có thể chọn ở ba chỉ số khác nhau.

**Quy tắc "nếu… thì…":**

- Nếu sắp xếp tăng dần được `s`, đáp án là `s[-1] + s[-2] - s[0]`.
- Với `n >= 3`, các chỉ số `0`, `n-2`, `n-1` của `s` đôi một khác nhau (vì `0 < n-2` khi `n >= 3`), nên ba phần tử này ở ba vị trí khác nhau.
- Nếu có giá trị trùng, vẫn đúng: chúng nằm ở các vị trí khác nhau trong mảng gốc.
- Không cần sắp xếp đủ: chỉ cần top-2 và min, tức một lượt duyệt.

**Invariant (một lượt):** sau khi xử lý `i` phần tử, `m1 >= m2` là hai giá trị lớn nhất của `nums[:i]`, và `lo` là giá trị nhỏ nhất.

In [9]:
# Kiểm chứng: s[-1] + s[-2] - s[0] bằng brute force trên mảng nhỏ
import random
from itertools import permutations
random.seed(3745)
for _ in range(500):
    n = random.randint(3, 8)
    nums = [random.randint(-10, 10) for _ in range(n)]
    s = sorted(nums)
    brute = max(nums[i] + nums[j] - nums[k] for i, j, k in permutations(range(n), 3))
    assert brute == s[-1] + s[-2] - s[0]
print("s[-1] + s[-2] - s[0] đúng")

s[-1] + s[-2] - s[0] đúng


## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp

Sắp xếp tăng dần rồi lấy `s[-1] + s[-2] - s[0]`. Ngắn gọn, chắc chắn đúng, `O(n log n)`.

In [10]:
class SolutionSort:
    def maximizeExpressionOfThree(self, nums):
        s = sorted(nums)
        return s[-1] + s[-2] - s[0]

print(SolutionSort().maximizeExpressionOfThree([1, 4, 2, 5]), SolutionSort().maximizeExpressionOfThree([-2, 0, 5, -2, 4]))

8 11


**Độ phức tạp:** Time `O(n log n)`, Space `O(n)`.

### Cách 3 — Một lượt duyệt theo dõi top-2 và min

Giữ `m1 >= m2` (hai lớn nhất) và `lo` (nhỏ nhất). Với mỗi `v`: cập nhật top-2 và min. Chỉ `O(n)` thời gian, `O(1)` bộ nhớ.

In [11]:
class SolutionOnePass:
    def maximizeExpressionOfThree(self, nums):
        NEG, POS = float('-inf'), float('inf')
        m1, m2, lo = NEG, NEG, POS
        for v in nums:
            if v > m1:
                m1, m2 = v, m1
            elif v > m2:
                m2 = v
            if v < lo:
                lo = v
        return m1 + m2 - lo

print(SolutionOnePass().maximizeExpressionOfThree([1, 4, 2, 5]), SolutionOnePass().maximizeExpressionOfThree([-2, 0, 5, -2, 4]))

8 11


**Độ phức tạp:** Time `O(n)`, Space `O(1)`. Chú ý: nếu `v == m1` ta không vào nhánh `v > m1` mà vào `v > m2`, nên giá trị trùng với `m1` vẫn được ghi vào `m2` (đúng vì là hai vị trí khác nhau).

### Cách 4 — `heapq`

`nlargest(2, nums)` cho hai lớn nhất và `nsmallest(1, nums)` cho nhỏ nhất. Rõ ý đồ, `O(n log k)` với `k` nhỏ.

In [12]:
import heapq

class SolutionHeap:
    def maximizeExpressionOfThree(self, nums):
        a, b = heapq.nlargest(2, nums)
        c = heapq.nsmallest(1, nums)[0]
        return a + b - c

print(SolutionHeap().maximizeExpressionOfThree([1, 4, 2, 5]), SolutionHeap().maximizeExpressionOfThree([-2, 0, 5, -2, 4]))

8 11


**Độ phức tạp:** Time `O(n)` (với `k = 2` và `1` hằng số), Space `O(1)`.

### Cách 5 — Lấy lần lượt rồi loại khỏi bản sao

Sao chép mảng; loại hai phần tử lớn nhất (mỗi lần `max` rồi `remove` **một** lần xuất hiện), rồi lấy `min` của phần còn lại làm `c`. Minh họa trực tiếp "ba chỉ số khác nhau", vì mỗi phần tử đã dùng bị loại đi.

In [13]:
class SolutionRemove:
    def maximizeExpressionOfThree(self, nums):
        rest = list(nums)
        a = max(rest); rest.remove(a)        # remove chỉ bỏ MỘT lần xuất hiện đầu tiên
        b = max(rest); rest.remove(b)
        c = min(rest)                        # min của phần còn lại (rest có >= 1 phần tử vì n >= 3)
        return a + b - c

print(SolutionRemove().maximizeExpressionOfThree([1, 4, 2, 5]), SolutionRemove().maximizeExpressionOfThree([-2, 0, 5, -2, 4]))

8 11


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (bản sao).

> Lưu ý: cách 5 chọn `c` là **min của phần còn lại**, có thể khác "min toàn mảng" trong lập luận, nhưng hai giá trị trùng nhau: `min` toàn mảng không nằm trong hai phần tử lớn nhất trừ khi tất cả bằng nhau, và khi đó giá trị cũng như nhau.

## 8. TRACE từng bước

Trace Cách 3 (một lượt) với `nums = [-2, 0, 5, -2, 4]`:

| v | `m1` | `m2` | `lo` | ghi chú |
|---|---|---|---|---|
| (đầu) | -inf | -inf | +inf | khởi tạo |
| -2 | -2 | -inf | -2 | `v > m1` |
| 0 | 0 | -2 | -2 | `v > m1`, đẩy `m1` xuống `m2` |
| 5 | 5 | 0 | -2 | `v > m1` |
| -2 | 5 | 0 | -2 | không vào top-2, không nhỏ hơn `lo` |
| 4 | 5 | 4 | -2 | `v > m2` |

Kết quả `5 + 4 - (-2) = 11`.

In [14]:
def trace_one_pass(nums):
    NEG, POS = float('-inf'), float('inf')
    m1, m2, lo = NEG, NEG, POS
    print("nums =", nums)
    for v in nums:
        note = []
        if v > m1:
            m1, m2 = v, m1
            note.append("v > m1: m1 <- v, m2 <- m1 cũ")
        elif v > m2:
            m2 = v
            note.append("v > m2: m2 <- v")
        if v < lo:
            lo = v
            note.append("v < lo")
        print(f"  v={v:>3}  m1={m1}  m2={m2}  lo={lo}  {'; '.join(note) or '(không đổi)'}")
    print(f"  kết quả = {m1} + {m2} - {lo} = {m1 + m2 - lo}")
    return m1 + m2 - lo

assert trace_one_pass([-2, 0, 5, -2, 4]) == 11
assert trace_one_pass([1, 4, 2, 5]) == 8

nums = [-2, 0, 5, -2, 4]
  v= -2  m1=-2  m2=-inf  lo=-2  v > m1: m1 <- v, m2 <- m1 cũ; v < lo
  v=  0  m1=0  m2=-2  lo=-2  v > m1: m1 <- v, m2 <- m1 cũ
  v=  5  m1=5  m2=0  lo=-2  v > m1: m1 <- v, m2 <- m1 cũ
  v= -2  m1=5  m2=0  lo=-2  (không đổi)
  v=  4  m1=5  m2=4  lo=-2  v > m2: m2 <- v
  kết quả = 5 + 4 - -2 = 11
nums = [1, 4, 2, 5]
  v=  1  m1=1  m2=-inf  lo=1  v > m1: m1 <- v, m2 <- m1 cũ; v < lo
  v=  4  m1=4  m2=1  lo=1  v > m1: m1 <- v, m2 <- m1 cũ
  v=  2  m1=4  m2=2  lo=1  v > m2: m2 <- v
  v=  5  m1=5  m2=4  lo=1  v > m1: m1 <- v, m2 <- m1 cũ
  kết quả = 5 + 4 - 1 = 8


In [15]:
def trace_remove(nums):
    rest = list(nums)
    print("ban đầu:", rest)
    a = max(rest); rest.remove(a); print(f"  chọn a={a}, còn {rest}")
    b = max(rest); rest.remove(b); print(f"  chọn b={b}, còn {rest}")
    c = min(rest); print(f"  chọn c={c} (nhỏ nhất của phần còn lại)")
    return a + b - c

assert trace_remove([1, 4, 2, 5]) == 8

ban đầu: [1, 4, 2, 5]
  chọn a=5, còn [1, 4, 2]
  chọn b=4, còn [1, 2]
  chọn c=1 (nhỏ nhất của phần còn lại)


## 9. Code nộp LeetCode

In [16]:
from typing import List, Optional

class Solution:
    def maximizeExpressionOfThree(self, nums: List[int]) -> int:
        m1 = m2 = float('-inf')
        lo = float('inf')
        for v in nums:
            if v > m1:
                m1, m2 = v, m1
            elif v > m2:
                m2 = v
            if v < lo:
                lo = v
        return m1 + m2 - lo

## 10. Vì sao thuật toán đúng

- **Tham lam trên hai nhóm độc lập:** `a + b - c` tăng khi `a`, `b` tăng và khi `c` giảm. Gọi `s` là mảng sắp xếp tăng. Với bất kỳ bộ ba chỉ số khác nhau nào, `a + b <= s[-1] + s[-2]` (tổng của hai phần tử bất kỳ ở hai chỉ số khác nhau không vượt tổng hai lớn nhất) và `c >= s[0]` (mọi phần tử `>=` nhỏ nhất). Suy ra `a + b - c <= s[-1] + s[-2] - s[0]`.
- **Đạt được:** `s[0]`, `s[-2]`, `s[-1]` ở ba chỉ số đôi một khác nhau của `s` vì `n >= 3` (nên `0 < n - 2 < n - 1`). Mỗi phần tử của `s` ứng với một vị trí riêng của `nums`, nên ba giá trị này nằm ở ba chỉ số khác nhau của `nums`. Do đó cận trên đạt được, và đó là đáp án.
- **Cách một lượt:** `m1, m2` luôn là hai phần tử lớn nhất đã thấy (kể cả khi trùng giá trị), và `lo` là nhỏ nhất, nên cuối vòng bằng `s[-1], s[-2], s[0]`.
- **Số âm:** khởi tạo bằng `±inf` đảm bảo phần tử đầu tiên luôn cập nhật được, kể cả khi mọi phần tử đều âm.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| Brute force ba vòng | `O(n^3)` | `O(1)` |
| Sắp xếp | `O(n log n)` | `O(n)` |
| Một lượt top-2 + min | `O(n)` | `O(1)` |
| `heapq` | `O(n)` (k hằng số) | `O(1)` |
| Loại khỏi bản sao | `O(n)` | `O(n)` |

`n = len(nums)`, `3 <= n <= 100`. Một lượt duyệt là tối ưu vì phải đọc mọi phần tử. Kết quả lớn nhất có thể là `100 + 100 - (-100) = 300`; nhỏ nhất xảy ra khi mọi phần tử bằng `-100`, cho `-100 + -100 - (-100) = -100`. Cả hai nằm gọn trong `int`.

## 12. Bộ test

Ví dụ của đề, các biên (`n = 3`, mọi phần tử bằng nhau, toàn âm, có trùng giá trị, `n = 100`, giá trị `±100`) và so sánh **tất cả cách** trên input ngẫu nhiên hợp lệ.

In [17]:
impls = [SolutionBrute(), SolutionSort(), SolutionOnePass(), SolutionHeap(), SolutionRemove(), Solution()]

cases = [
    ([1, 4, 2, 5], 8), ([-2, 0, 5, -2, 4], 11), ([3, 3, 3], 3), ([-1, -2, -3], 0),
    ([1, 2, 3], 4), ([3, 2, 1], 4), ([-100, -100, -100], -100), ([100, 100, 100], 100),
    ([100, 100, -100], 300), ([5, 5, 1, 1], 9), ([0, 0, 0, 0], 0), ([-5, -4, -3, -2, -1], 2),
]
for nums, exp in cases:
    assert 3 <= len(nums) <= 100 and all(-100 <= v <= 100 for v in nums)
    for impl in impls:
        got = impl.maximizeExpressionOfThree(list(nums))
        assert got == exp, (type(impl).__name__, nums, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 12 ca


In [18]:
import random
from itertools import permutations
random.seed(37450)
for _ in range(400):                                  # so sánh với brute force trên mảng nhỏ
    n = random.randint(3, 9)
    nums = [random.randint(-6, 6) for _ in range(n)]
    exp = max(nums[i] + nums[j] - nums[k] for i, j, k in permutations(range(n), 3))
    outs = [impl.maximizeExpressionOfThree(list(nums)) for impl in impls]
    assert all(o == exp for o in outs), (nums, outs, exp)
for _ in range(500):                                  # mảng lớn: bỏ brute force
    n = random.randint(3, 100)
    nums = [random.randint(-100, 100) for _ in range(n)]
    s = sorted(nums)
    exp = s[-1] + s[-2] - s[0]
    outs = [impl.maximizeExpressionOfThree(list(nums)) for impl in impls[1:]]
    assert all(o == exp for o in outs), (nums, outs, exp)
print("900 test ngẫu nhiên (seed=37450): 6 cách (brute chỉ trên mảng nhỏ) giống nhau")

900 test ngẫu nhiên (seed=37450): 6 cách (brute chỉ trên mảng nhỏ) giống nhau


In [19]:
# Tính chất: kết quả không đổi khi xáo trộn; không nhỏ hơn bất kỳ bộ ba cụ thể nào
for _ in range(200):
    n = random.randint(3, 30)
    nums = [random.randint(-100, 100) for _ in range(n)]
    r = Solution().maximizeExpressionOfThree(nums)
    i, j, k = random.sample(range(n), 3)
    assert r >= nums[i] + nums[j] - nums[k]
    random.shuffle(nums)
    assert r == Solution().maximizeExpressionOfThree(nums)
print("bất biến theo thứ tự và là cận trên của mọi bộ ba OK")

bất biến theo thứ tự và là cận trên của mọi bộ ba OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng một phần tử hai lần (`2 * max - min`)

Chọn `a = b = max` vi phạm điều kiện "chỉ số khác nhau" khi chỉ có **một** phần tử lớn nhất.

In [20]:
def wrong_reuse_max(nums):
    return 2 * max(nums) - min(nums)          # SAI: dùng max hai lần

print(wrong_reuse_max([1, 4, 2, 5]), "!=", Solution().maximizeExpressionOfThree([1, 4, 2, 5]))
assert wrong_reuse_max([1, 4, 2, 5]) == 9 != 8

9 != 8


### Lỗi 2: khởi tạo max bằng `0` khi có số âm

`nums[i]` có thể âm. Nếu khởi tạo `m1 = m2 = 0` thì với mảng toàn âm, `m1, m2` không bao giờ được cập nhật.

In [21]:
def wrong_init_zero(nums):
    m1 = m2 = 0                               # SAI: nên là -inf
    lo = float('inf')
    for v in nums:
        if v > m1:
            m1, m2 = v, m1
        elif v > m2:
            m2 = v
        lo = min(lo, v)
    return m1 + m2 - lo

print(wrong_init_zero([-1, -2, -3]), "!=", Solution().maximizeExpressionOfThree([-1, -2, -3]))
assert wrong_init_zero([-1, -2, -3]) == 3 != 0

3 != 0


### Lỗi 3: cộng `c` thay vì trừ (hoặc chọn `c` lớn)

Biểu thức là `a + b - c`; muốn lớn nhất thì `c` phải **nhỏ nhất**, không phải lớn nhất. Cộng ba số lớn nhất là một bài khác.

In [22]:
def wrong_top3_sum(nums):
    s = sorted(nums)
    return s[-1] + s[-2] + s[-3]              # SAI: bài khác (tổng ba lớn nhất)

print(wrong_top3_sum([1, 4, 2, 5]), "!=", Solution().maximizeExpressionOfThree([1, 4, 2, 5]))
assert wrong_top3_sum([1, 4, 2, 5]) == 11 != 8

11 != 8


### Lỗi 4: dùng `set` để "khử trùng"

Đề chỉ yêu cầu **chỉ số** khác nhau, giá trị trùng vẫn hợp lệ. Dùng `set` làm mất các bản sao: mảng `[5, 5, 1, 1]` chỉ còn hai giá trị, và mảng `[1, 2, 3, 3]` bị tính thành `3 + 2 - 1 = 4` thay vì `3 + 3 - 1 = 5`.

In [23]:
def wrong_use_set(nums):
    s = sorted(set(nums))                      # SAI: set làm mất giá trị trùng
    return s[-1] + s[-2] - s[0] if len(s) >= 3 else None

print(wrong_use_set([5, 5, 1, 1]), "vs", Solution().maximizeExpressionOfThree([5, 5, 1, 1]))
assert wrong_use_set([5, 5, 1, 1]) is None and Solution().maximizeExpressionOfThree([5, 5, 1, 1]) == 9
assert wrong_use_set([1, 2, 3, 3]) == 4 != Solution().maximizeExpressionOfThree([1, 2, 3, 3]) == 5

None vs 9


### Lỗi 5: đảo thứ tự khi sắp xếp giảm dần rồi dùng chỉ số tăng

Sắp xếp giảm dần (`reverse=True`) nhưng vẫn dùng `s[-1] + s[-2] - s[0]` làm đảo ngược vai trò lớn/nhỏ.

In [24]:
def wrong_desc_index(nums):
    s = sorted(nums, reverse=True)
    return s[-1] + s[-2] - s[0]               # SAI: với s giảm dần thì s[0] mới là lớn nhất

print(wrong_desc_index([1, 4, 2, 5]), "!=", Solution().maximizeExpressionOfThree([1, 4, 2, 5]))
assert wrong_desc_index([1, 4, 2, 5]) == 1 + 2 - 5 == -2 != 8

-2 != 8


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Biểu thức "cộng một số, trừ một số" -> tham lam: số được cộng lớn nhất, số bị trừ nhỏ nhất.
- "Chọn ở các chỉ số khác nhau" -> kiểm tra `n` đủ lớn để ba phần tử cực trị không trùng vị trí; `n >= 3` đảm bảo.
- Luôn dùng `±inf` khi có số âm; tuyệt đối không khởi tạo `0`.
- Nhận ra đây là "top-k" -> sắp xếp (dễ) hoặc một lượt (tối ưu) hoặc `heapq`.
- Một dòng nộp nhanh: `s = sorted(nums); return s[-1] + s[-2] - s[0]`.
- Test nhanh bằng mảng độ dài `3`: có đúng một cách chọn bộ ba (theo giá trị) nên dễ tính tay.

## 15. Biến thể của bài

1. **Maximum Product of Three Numbers** (LeetCode 628): tích lớn nhất của ba số; phải xét cả hai số âm lớn nhất.
2. **Maximum Product of Two Elements in an Array** (1464): `(a-1)*(b-1)` lớn nhất.
3. **Third Maximum Number** (414): số lớn thứ ba **khác nhau** (phân biệt với bài này, nơi giá trị trùng được tính riêng).
4. **Largest Perimeter Triangle** (976): sắp xếp giảm dần, tìm ba cạnh liên tiếp tạo tam giác.
5. **Maximize `a - b + c - d`** trên bốn phần tử khác nhau: hai lớn, hai nhỏ.
6. **Kth largest** tổng quát: `heapq.nlargest(k, nums)` hoặc quickselect.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Brute force ba vòng | thử mọi bộ ba chỉ số | `O(n^3)` | `O(1)` | Oracle kiểm tra |
| Sắp xếp | `s[-1] + s[-2] - s[0]` | `O(n log n)` | `O(n)` | Nộp nhanh, dễ chứng minh |
| Một lượt | giữ `m1, m2, lo` | `O(n)` | `O(1)` | Tối ưu, nộp bài |
| `heapq` | `nlargest(2)` + `nsmallest(1)` | `O(n)` | `O(1)` | Rõ ý đồ, thư viện chuẩn |
| Loại khỏi bản sao | `max`, `remove`, rồi `min` | `O(n)` | `O(n)` | Minh họa "chỉ số khác nhau" |

Nộp cách một lượt (hoặc sắp xếp nếu muốn ngắn).

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().maximizeExpressionOfThree([10, -10, 3])` ra gì? `Solution().maximizeExpressionOfThree([7, 7, 7, 7])` ra gì?

**Bài 2 (tính tay):** với `nums = [4, -1, 9, 0, 9]`, xác định `a, b, c` và kết quả.

**Bài 3 (điền TODO):** hoàn thành `max_product_of_three(nums)` (LeetCode 628): tích lớn nhất của ba phần tử ở ba chỉ số khác nhau.

**Bài 4 (tự viết lại):** viết `maximize_four(nums)` cho `a + b - c - d` lớn nhất trên bốn chỉ số khác nhau (`n >= 4`).

**Bài 5 (mini challenge):** viết `minimize_expression(nums)` cho `a + b - c` **nhỏ nhất**, rồi kiểm tra bằng brute force.

In [25]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().maximizeExpressionOfThree([10, -10, 3]), Solution().maximizeExpressionOfThree([7, 7, 7, 7]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def max_product_of_three_todo(nums):
    s = sorted(nums)
    # TODO: so sánh (ba lớn nhất) với (hai nhỏ nhất âm * lớn nhất)
    return 0

23 7


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `[10, -10, 3]` chỉ có một cách chọn tập ba phần tử; thử đặt `-10` làm `c`. Bài 2: hai lớn nhất là hai số `9`.

**Mức 2 (vừa):** Bài 3: tích ba số lớn nhất, hoặc tích của hai số nhỏ nhất (có thể là hai số âm) với số lớn nhất. Bài 4: lấy hai lớn nhất và hai nhỏ nhất của mảng đã sắp xếp, `n >= 4` đảm bảo bốn chỉ số khác nhau. Bài 5: lấy `a, b` nhỏ nhất và `c` lớn nhất.

**Mức 3 (gần đáp án):** Bài 1: `10 + 3 - (-10) = 23` và `7`. Bài 2: `9 + 9 - (-1) = 19`. Bài 3: `max(s[-1]*s[-2]*s[-3], s[0]*s[1]*s[-1])`. Bài 4: `s[-1] + s[-2] - s[0] - s[1]`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
from itertools import permutations, combinations

# Bài 1, 2
assert Solution().maximizeExpressionOfThree([10, -10, 3]) == 23
assert Solution().maximizeExpressionOfThree([7, 7, 7, 7]) == 7
assert Solution().maximizeExpressionOfThree([4, -1, 9, 0, 9]) == 19

# Bài 3
def max_product_of_three(nums):
    s = sorted(nums)
    return max(s[-1] * s[-2] * s[-3], s[0] * s[1] * s[-1])
assert max_product_of_three([1, 2, 3]) == 6 and max_product_of_three([-10, -10, 5, 2]) == 500
assert max_product_of_three([-1, -2, -3]) == -6
for _ in range(200):
    n = random.randint(3, 8)
    nums = [random.randint(-10, 10) for _ in range(n)]
    assert max_product_of_three(nums) == max(nums[i] * nums[j] * nums[k] for i, j, k in combinations(range(n), 3))

# Bài 4
def maximize_four(nums):
    s = sorted(nums)
    return s[-1] + s[-2] - s[0] - s[1]
for _ in range(200):
    n = random.randint(4, 8)
    nums = [random.randint(-10, 10) for _ in range(n)]
    brute = max(nums[a] + nums[b] - nums[c] - nums[d] for a, b, c, d in permutations(range(n), 4))
    assert maximize_four(nums) == brute

# Bài 5
def minimize_expression(nums):
    s = sorted(nums)
    return s[0] + s[1] - s[-1]
for _ in range(200):
    n = random.randint(3, 8)
    nums = [random.randint(-10, 10) for _ in range(n)]
    assert minimize_expression(nums) == min(nums[i] + nums[j] - nums[k] for i, j, k in permutations(range(n), 3))
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "chọn cực trị ở các chỉ số khác nhau":**

```python
s = sorted(nums)
return s[-1] + s[-2] - s[0]          # hai lớn nhất, một nhỏ nhất (n >= 3)
# một lượt: giữ m1 >= m2 (hai lớn nhất) và lo (nhỏ nhất), khởi tạo bằng ±inf
```

**Checklist trước khi submit:**

- [ ] `a, b` là hai **lớn nhất**, `c` là **nhỏ nhất** (không đảo vai trò)?
- [ ] Ba phần tử ở ba chỉ số khác nhau (không dùng `max` hai lần, không dùng `set`)?
- [ ] Khởi tạo bằng `±inf`, không phải `0`, để xử lý mảng toàn âm?
- [ ] Giá trị trùng nhau vẫn được tính ở các vị trí khác nhau?
- [ ] Thử `[3,3,3]`, `[-1,-2,-3]` và hai ví dụ của đề?
- [ ] Chữ ký đúng `maximizeExpressionOfThree(self, nums: List[int]) -> int`?